# 🦀 Day 4: Error Handling & Edge Cases
---

Today we make RustKV robust with comprehensive error handling.

- **Custom error types**: `KvError` enum with `thiserror`
- **Error categories**: Protocol, storage, IO, auth
- **Converting errors**: `From` implementations
- **Graceful degradation**: Disk full, network drops
- **Input validation**: Key length, value size limits
- **Exercise**: Add rate limiting with proper error responses

## Custom error type with thiserror

Add to Cargo.toml: `thiserror = "1"`

In [ ]:
:dep thiserror = "1"

use thiserror::Error;

#[derive(Error, Debug)]
pub enum KvError {
    #[error("Protocol error: {0}")]
    Protocol(String),

    #[error("Storage error: {0}")]
    Storage(String),

    #[error("IO error: {0}")]
    Io(#[from] std::io::Error),

    #[error("Auth error: {0}")]
    Auth(String),

    #[error("Wrong type: {0}")]
    WrongType(String),

    #[error("Key too long (max {max} bytes)")]
    KeyTooLong { max: usize },

    #[error("Value too large (max {max} bytes)")]
    ValueTooLarge { max: usize },
}

impl KvError {
    pub fn protocol(msg: impl Into<String>) -> Self { KvError::Protocol(msg.into()) }
    pub fn storage(msg: impl Into<String>) -> Self { KvError::Storage(msg.into()) }
}

println!("KvError: {:?}", KvError::KeyTooLong { max: 512 });

In [ ]:
// Input validation — key length and value size limits

const MAX_KEY_LEN: usize = 512;
const MAX_VALUE_LEN: usize = 512 * 1024;  // 512 KB

pub fn validate_key(key: &str) -> Result<(), KvError> {
    if key.len() > MAX_KEY_LEN {
        Err(KvError::KeyTooLong { max: MAX_KEY_LEN })
    } else if key.is_empty() {
        Err(KvError::Protocol("empty key not allowed".to_string()))
    } else {
        Ok(())
    }
}

pub fn validate_value_size(len: usize) -> Result<(), KvError> {
    if len > MAX_VALUE_LEN {
        Err(KvError::ValueTooLarge { max: MAX_VALUE_LEN })
    } else {
        Ok(())
    }
}

// Edge cases: empty keys, binary data (allow), concurrent deletes (return (nil) or error)
assert!(validate_key("").is_err());
assert!(validate_key("ok").is_ok());
println!("Validation: empty key rejected, valid key OK.");

In [ ]:
// Converting to protocol response — map KvError to client-facing string

impl KvError {
    pub fn to_protocol_response(&self) -> String {
        match self {
            KvError::Protocol(msg) => format!("-ERR {}\r\n", msg),
            KvError::Storage(msg) => format!("-ERR storage: {}\r\n", msg),
            KvError::WrongType(msg) => format!("-WRONGTYPE {}\r\n", msg),
            KvError::KeyTooLong { max } => format!("-ERR key too long (max {} bytes)\r\n", max),
            KvError::ValueTooLarge { max } => format!("-ERR value too large (max {} bytes)\r\n", max),
            KvError::Auth(msg) => format!("-NOAUTH {}\r\n", msg),
            KvError::Io(e) => format!("-ERR io: {}\r\n", e),
        }
    }
}

let err = KvError::KeyTooLong { max: 512 };
println!("Response: {}", err.to_protocol_response());

In [ ]:
// Error handling tests

#[cfg(test)]
mod tests {
    use super::*;

    #[test]
    fn test_validate_key_empty() {
        assert!(validate_key("").is_err());
    }

    #[test]
    fn test_validate_key_ok() {
        assert!(validate_key("mykey").is_ok());
    }

    #[test]
    fn test_protocol_response() {
        let e = KvError::Protocol("bad command".to_string());
        assert!(e.to_protocol_response().starts_with("-ERR"));
    }
}

println!("Error handling tests defined. Run: cargo test");

## 📝 Exercise: Add rate limiting with proper error responses

Track requests per client (e.g., per IP or connection ID). When limit exceeded, return a clear error like `-ERR too many requests, try again later\r\n`. Use a sliding window or token bucket.

In [ ]:
// YOUR CODE HERE
// Add KvError::RateLimitExceeded variant
// In handle_client: check rate before executing command
// Return to_protocol_response() for rate limit errors

// struct RateLimiter { requests: AtomicUsize, window_start: Instant, max_per_sec: u32 }

println!("Implement rate limiting with KvError::RateLimitExceeded.");

## 🎯 Key Takeaways

1. Use `thiserror` for clean, `#[derive(Error)]` enum with `#[error(...)]` messages
2. Error categories: Protocol, Storage, Io, Auth, WrongType, validation errors
3. `From<std::io::Error>` lets `?` convert IO errors automatically
4. Input validation: key length, value size, empty keys — fail fast with clear errors
5. `to_protocol_response()` maps internal errors to client-facing RESP strings

---
**Tomorrow:** Testing